# HandsOn HO-05-GraphInDS
Welcome to handsOn HO-05-GraphInDS: graph in distributed system, using GraphFrames. To use GraphFrames, you need to run ```pyspark --packages graphframes:graphframes:0.8.0-spark2.4-s_2.11``` (adjust with your Spark version. That script uses Spark 2.4.x, the one used in our VM, and it will download the graphframes package automatically, thus, make sure you have internet connection when launching the script).

Note: you can use any Spark & GraphFrames API (without building from-the-scratch).

import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .appName("HO05-GraphInDS")
    .getOrCreate()
 )

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

In [ ]:
from pyspark.sql.types import *
from graphframes import *
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql.functions import col, count, sum as fsum, lit, coalesce, when, avg


def create_transport_graph(nodes_filePath, edges_filePath):
    node_fields = [
        StructField("id", StringType(), True),
        StructField("latitude", FloatType(), True),
        StructField("longitude", FloatType(), True),
        StructField("population", IntegerType(), True)
    ]
    nodes = spark.read.csv(nodes_filePath, header=True,
                           schema=StructType(node_fields))
    edge_fields = [
        StructField("src", StringType(), True),
        StructField("dst", StringType(), True),
        StructField("relationship", StringType(), True),
        StructField("cost", FloatType(), True)
    ]
    rels = spark.read.csv(edges_filePath, header=True,
                         schema=StructType(edge_fields))
    reversed_rels = (rels.withColumn("newSrc", rels.dst)
                     .withColumn("newDst", rels.src)
                     .drop("dst", "src")
                     .withColumnRenamed("newSrc", "src")
                     .withColumnRenamed("newDst", "dst")
                     .select("src", "dst", "relationship", "cost"))

    relationships = rels.union(reversed_rels)

    return GraphFrame(nodes, relationships)

g = create_transport_graph("./transport-nodes.csv", "./transport-relationships.csv")
g.vertices.show()
g.edges.show()

+----------------+---------+---------+----------+
|              id| latitude|longitude|population|
+----------------+---------+---------+----------+
|       Amsterdam| 52.37919| 4.899431|    821752|
|         Utrecht|52.092876|  5.10448|    334176|
|        Den Haag|52.078663| 4.288788|    514861|
|       Immingham| 53.61239| -0.22219|      9642|
|       Doncaster| 53.52285| -1.13116|    302400|
|Hoek van Holland|  51.9775|  4.13333|      9382|
|      Felixstowe| 51.96375|   1.3511|     23689|
|         Ipswich| 52.05917|  1.15545|    133384|
|      Colchester| 51.88921|  0.90421|    104390|
|          London|51.509865|-0.118092|   8787892|
|       Rotterdam|  51.9225|  4.47917|    623652|
|           Gouda| 52.01667|  4.70833|     70939|
+----------------+---------+---------+----------+

+----------------+----------------+------------+-----+
|             src|             dst|relationship| cost|
+----------------+----------------+------------+-----+
|       Amsterdam|         Utrecht

In [ ]:
edge_fields = [
    StructField("src", StringType(), True),
    StructField("dst", StringType(), True),
    StructField("relationship", StringType(), True),
    StructField("cost", FloatType(), True)
]
rels = spark.read.csv("./transport-relationships.csv", header=True, schema=StructType(edge_fields))

# buat balikannya soal nya a->b, b-> a
reversed_rels = (
    rels.withColumn("newSrc", rels.dst)
        .withColumn("newDst", rels.src)
        .drop("dst", "src")
        .withColumnRenamed("newSrc", "src")
        .withColumnRenamed("newDst", "dst")
        .select("src", "dst", "relationship", "cost")
)
edges = rels.union(reversed_rels)

max_cost = edges.agg({"cost": "max"}).first()[0]
min_cost = edges.agg({"cost": "min"}).first()[0]

edges.filter(col("cost") == max_cost).select("src", "dst", "cost").show(truncate=False)
edges.filter(col("cost") == min_cost).select("src", "dst", "cost").show(truncate=False)
edges.agg(avg("cost").alias("average_cost")).show(truncate=False)

Maximum cost pair(s):
+---------+---------+-----+
|src      |dst      |cost |
+---------+---------+-----+
|Amsterdam|Immingham|369.0|
|Immingham|Amsterdam|369.0|
+---------+---------+-----+

Minimum cost pair(s):
+----------+----------+----+
|src       |dst       |cost|
+----------+----------+----+
|Ipswich   |Felixstowe|22.0|
|Felixstowe|Ipswich   |22.0|
+----------+----------+----+

Average cost:
+-----------------+
|average_cost     |
+-----------------+
|91.33333333333333|
+-----------------+



### Milestone 02
1. Find flight routes that have no direct connection

In [ ]:
ab = edges.selectExpr("src as a", "dst as b")
bc = edges.selectExpr("src as b", "dst as c")
direct = edges.selectExpr("src as a", "dst as c")

result = (
    ab.join(bc, "b")
        # filter untuk menghindari kasus a->b->a
      .filter("a <> c")
         # filter untuk menghindari kasus a->b->c yang sudah ada a->c
      .join(direct, ["a", "c"], "left_anti")
      # biar no duplikat
      .distinct()
      .orderBy("a", "b", "c")
)

result.show(truncate=False)

+----------+----------------+----------------+
|a_id      |c_id            |b_id            |
+----------+----------------+----------------+
|Amsterdam |Gouda           |Den Haag        |
|Amsterdam |Hoek van Holland|Den Haag        |
|Amsterdam |Rotterdam       |Den Haag        |
|Amsterdam |Doncaster       |Immingham       |
|Amsterdam |Gouda           |Utrecht         |
|Colchester|Felixstowe      |Ipswich         |
|Colchester|Doncaster       |London          |
|Den Haag  |Immingham       |Amsterdam       |
|Den Haag  |Utrecht         |Amsterdam       |
|Den Haag  |Utrecht         |Gouda           |
|Den Haag  |Felixstowe      |Hoek van Holland|
|Doncaster |Amsterdam       |Immingham       |
|Doncaster |Colchester      |London          |
|Felixstowe|Den Haag        |Hoek van Holland|
|Felixstowe|Rotterdam       |Hoek van Holland|
|Felixstowe|Colchester      |Ipswich         |
|Gouda     |Amsterdam       |Den Haag        |
|Gouda     |Hoek van Holland|Den Haag        |
|Gouda     |H

### Milestone 03
1. Find the most important airport (you can use any measurement to judge the importance level of airports, and please describe why you choose it -the measurement-)

In [29]:
beta = 0.8
num_iters = 10

# semua src atau dst jadi id
nodes_df = (
    edges.selectExpr("src as id")
         .union(edges.selectExpr("dst as id"))
         .distinct()
 )
# jumlah panah keluar dari setiap node
out_deg = edges.groupBy("src").agg(count("*").alias("out_degree"))

# inisialisasi rank 1/n
n = nodes_df.count()
ranks = nodes_df.withColumn("rank", lit(1.0 / n))

for _ in range(num_iters):
    contribs = (
        edges.alias("e")
        .join(ranks.alias("r"), col("e.src") == col("r.id"), "left")
        .join(out_deg.alias("o"), col("e.src") == col("o.src"), "left")
        .select(
            col("e.dst").alias("id"),
            when(col("o.out_degree") > 0, col("r.rank") / col("o.out_degree")).otherwise(lit(0.0)).alias("contrib")
        )
    )

    incoming = contribs.groupBy("id").agg(fsum("contrib").alias("incoming_rank"))

    dangling_mass_row = (
        ranks.alias("r")
        .join(out_deg.alias("o"), col("r.id") == col("o.src"), "left_anti")
        .agg(fsum("r.rank").alias("dangling_mass"))
        .first()
    )
    dangling_mass = dangling_mass_row["dangling_mass"] if dangling_mass_row and dangling_mass_row["dangling_mass"] is not None else 0.0

    ranks = (
        nodes_df.alias("n")
        .join(incoming.alias("i"), col("n.id") == col("i.id"), "left")
        .select(
            col("n.id").alias("id"),
            (
                lit((1.0 - beta) / n)
                + lit(beta) * (coalesce(col("i.incoming_rank"), lit(0.0)) + lit(dangling_mass / n))
            ).alias("rank")
        )
    )

importance = ranks.orderBy(col("rank").desc())
importance.show(truncate=False)

most_important = importance.first()

+----------------+-------------------+
|id              |rank               |
+----------------+-------------------+
|Den Haag        |0.11328784096422519|
|Amsterdam       |0.09455567547505292|
|Hoek van Holland|0.09123414543887731|
|Gouda           |0.08862619170564559|
|Rotterdam       |0.08730987423523288|
|London          |0.07970964161999695|
|Colchester      |0.07947196128850276|
|Doncaster       |0.07768813054855458|
|Ipswich         |0.07725910738994393|
|Immingham       |0.07318960919372047|
|Felixstowe      |0.07199010305289053|
|Utrecht         |0.06567771908735684|
+----------------+-------------------+



Menggunakan metode page rank untuk menentukan airport terpenting, metode ini dipilih karena penerbangan dari airport yang penting juga seharusnya memililki pengaruh terhadap ranking, tidak hanya jumlah penerbangan yang dilayani.

### Milestone 04
1. Find the sortest path based on "node" -path with fewest nodes- from Amsterdam to London

In [31]:
start = "Amsterdam"
end = "London"

# pake BFS
frontier = [(start, [start])]
visited = {start}
shortest_path = None

while frontier and shortest_path is None:
    next_frontier = []
    for node, path in frontier:
        neighbors = [
            r["dst"]
            for r in edges.filter(edges.src == node).select("dst").collect()
            if r["dst"] not in visited
        ]
        for nxt in neighbors:
            new_path = path + [nxt]
            if nxt == end:
                shortest_path = new_path
                break
            visited.add(nxt)
            next_frontier.append((nxt, new_path))
        if shortest_path is not None:
            break
    frontier = next_frontier

print("Shortest path by nodes:", " -> ".join(shortest_path))

Shortest path by nodes: Amsterdam -> Immingham -> Doncaster -> London


### Bonus
1. Find the sortest path based on "cost value" -see 'cost' column in the edge/relationship dataframe - from Amsterdam to London

In [32]:
import heapq
from collections import defaultdict

# pake dijkstra
graph = defaultdict(list)
for row in edges.select("src", "dst", "cost").collect():
    graph[row["src"]].append((row["dst"], float(row["cost"])))

start = "Amsterdam"
end = "London"
pq = [(0.0, start, [start])]
best_seen = {}
best_cost = None
best_path = None

while pq:
    cost, node, path = heapq.heappop(pq)
    if node in best_seen and cost >= best_seen[node]:
        continue
    best_seen[node] = cost
    if node == end:
        best_cost = cost
        best_path = path
        break
    for nxt, w in graph[node]:
        new_cost = cost + w
        if nxt not in best_seen or new_cost < best_seen[nxt]:
            heapq.heappush(pq, (new_cost, nxt, path + [nxt]))

print("Shortest path by cost:", " -> ".join(best_path))

Shortest path by cost: Amsterdam -> Den Haag -> Hoek van Holland -> Felixstowe -> Ipswich -> Colchester -> London


# Submission
Submit this ```ipynb``` file to the course portal, with format: ```HO_05_GraphInDS_NIM_NamaLengkap.ipynb```. Make sure when submitting this file, each code cell has the outputs (not blank).